# 02 — Build the Analytical Data Model

**Project:** Olist Brazilian E-Commerce — Delivery Reliability & Customer Experience  
**Stage:** Transform cleaned source tables into reusable analytical facts.

## Business purpose

Create two validated datasets:

1. **`order_level_analytics.csv`** — exactly one row per order. This is the source for order-level delivery KPIs and review comparisons.
2. **`order_item_analytics.csv`** — exactly one row per order item, enriched with order, product category, customer state and seller attributes. This supports category and seller/region analysis without forcing many-valued item attributes into the order table.

We aggregate items, payments and reviews separately **before** joining them to orders. This avoids join fan-out (for example, three item rows multiplied by two payment rows would create six rows for one order and inflate totals).

### Important business definitions

- **Late delivery** is evaluated on calendar dates: `actual_delivery_date > estimated_delivery_date`. Olist's estimated-delivery field often has a midnight timestamp, so comparing full timestamps can incorrectly classify a delivery that arrived later on the same calendar day as late.
- **Eligible delivery order** means status is `delivered`, purchase/actual/estimated dates are present, and actual and estimated dates are not before the purchase date.
- **Low-rated order** means the mean of its available review scores is `<= 2`. This is one documented definition; we can do sensitivity checks later.
- Missing order-item/payment records remain missing for monetary totals; they are not silently converted to zero. Count columns can be zero while monetary totals remain null.

Run Notebook 01 first with `EXPORT_CLEANED_TABLES = True`. The original data ZIP should be in `data/raw/`.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebooks" else CURRENT_DIR
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "analytical_model"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPORT_ANALYTIC_TABLES = True


## 1. Load the cleaned tables

This notebook reads the cleaned extracts created by Notebook 01. Identifier and postal-prefix columns are loaded as strings to protect IDs and leading zeros. Date fields are parsed explicitly so interval calculations behave consistently.


In [2]:
REQUIRED_PROCESSED_FILES = {
    "orders": "orders_clean.csv",
    "customers": "customers_clean.csv",
    "order_items": "order_items_clean.csv",
    "payments": "payments_clean.csv",
    "reviews": "reviews_clean.csv",
    "products": "products_clean.csv",
    "sellers": "sellers_clean.csv",
    "category_translation": "category_translation_clean.csv",
    "geolocation_lookup": "geolocation_prefix_lookup.csv",
}
missing_files = [name for name in REQUIRED_PROCESSED_FILES.values() if not (PROCESSED_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(
        "Cleaned input files are missing: " + ", ".join(missing_files) +
        ". Run notebooks/01_data_loading_cleaning.ipynb first with EXPORT_CLEANED_TABLES = True."
    )

STRING_DTYPES = {
    "orders": {"order_id": "string", "customer_id": "string", "order_status": "string"},
    "customers": {"customer_id": "string", "customer_unique_id": "string", "customer_zip_code_prefix": "string", "customer_city": "string", "customer_state": "string"},
    "order_items": {"order_id": "string", "product_id": "string", "seller_id": "string"},
    "payments": {"order_id": "string", "payment_type": "string"},
    "reviews": {"order_id": "string", "review_id": "string"},
    "products": {"product_id": "string", "product_category_name": "string"},
    "sellers": {"seller_id": "string", "seller_zip_code_prefix": "string", "seller_city": "string", "seller_state": "string"},
    "category_translation": {"product_category_name": "string", "product_category_name_english": "string"},
    "geolocation_lookup": {"geolocation_zip_code_prefix": "string", "geolocation_city": "string", "geolocation_state": "string"},
}

tables = {}
for table_name, filename in REQUIRED_PROCESSED_FILES.items():
    tables[table_name] = pd.read_csv(PROCESSED_DIR / filename, dtype=STRING_DTYPES.get(table_name, {}), low_memory=False)

ORDER_DATE_COLUMNS = [
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
]
for column in ORDER_DATE_COLUMNS:
    tables["orders"][column] = pd.to_datetime(tables["orders"][column], errors="coerce")
tables["order_items"]["shipping_limit_date"] = pd.to_datetime(tables["order_items"]["shipping_limit_date"], errors="coerce")
for column in ["price", "freight_value"]:
    tables["order_items"][column] = pd.to_numeric(tables["order_items"][column], errors="coerce")
for column in ["payment_value", "payment_installments"]:
    tables["payments"][column] = pd.to_numeric(tables["payments"][column], errors="coerce")
tables["reviews"]["review_score"] = pd.to_numeric(tables["reviews"]["review_score"], errors="coerce")
for column in ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]:
    if column in tables["products"].columns:
        tables["products"][column] = pd.to_numeric(tables["products"][column], errors="coerce")
for column in ["geolocation_lat", "geolocation_lng"]:
    tables["geolocation_lookup"][column] = pd.to_numeric(tables["geolocation_lookup"][column], errors="coerce")

input_profile = pd.DataFrame([
    {"table": name, "rows": len(df), "columns": len(df.columns), "memory_mb": round(df.memory_usage(deep=True).sum()/1024**2, 2)}
    for name, df in tables.items()
]).sort_values("table").reset_index(drop=True)
display(input_profile)


,table,rows,columns,memory_mb
0,category_translation,71,2,0.01
1,customers,99441,5,30.95
2,geolocation_lookup,19015,17,4.51
3,order_items,112650,7,29.54
4,orders,99441,8,24.66
5,payments,103886,5,16.23
6,products,32951,9,6.31
7,reviews,99224,7,39.11
8,sellers,3095,4,0.72


## 2. Validate grains before joining

We check the expected grain of each source table first. If these tests fail, do not proceed with aggregations until the reason is understood.


In [3]:
# Expected grain: one order; one order-item line; one row per payment sequence; one row per review/order pair.
assert tables["orders"]["order_id"].notna().all(), "orders.order_id has nulls"
assert tables["orders"]["order_id"].is_unique, "orders is not one row per order_id"
assert tables["customers"]["customer_id"].notna().all() and tables["customers"]["customer_id"].is_unique
assert not tables["order_items"].duplicated(["order_id", "order_item_id"]).any(), "Duplicate order-item key"
assert not tables["payments"].duplicated(["order_id", "payment_sequential"]).any(), "Duplicate payment key"
assert not tables["reviews"].duplicated(["review_id", "order_id"]).any(), "Duplicate review/order pair"
assert tables["products"]["product_id"].is_unique
assert tables["sellers"]["seller_id"].is_unique
assert tables["category_translation"]["product_category_name"].is_unique
assert tables["geolocation_lookup"]["geolocation_zip_code_prefix"].notna().all()
assert tables["geolocation_lookup"]["geolocation_zip_code_prefix"].is_unique

source_counts = pd.DataFrame([
    {"table": name, "rows": len(tables[name]), "distinct_order_ids": int(tables[name]["order_id"].nunique(dropna=True)) if "order_id" in tables[name].columns else pd.NA}
    for name in ["orders", "order_items", "payments", "reviews"]
])
display(source_counts)


,table,rows,distinct_order_ids
0,orders,99441,99441
1,order_items,112650,98666
2,payments,103886,99440
3,reviews,99224,98673


## 3. Build customer and seller dimensions with safe geolocation joins

The geolocation lookup has one row per postal-code prefix. We use validated many-to-one **left joins** so a missing geolocation match cannot remove a valid customer, seller or item row.


In [4]:
geo = tables["geolocation_lookup"].copy()
geo_columns = [c for c in [
    "geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng", "geolocation_city", "geolocation_state",
    "has_city_label_conflict", "has_state_label_conflict", "has_missing_coordinates", "has_out_of_range_coordinates"
] if c in geo.columns]
geo = geo[geo_columns].copy()

customer_dim = tables["customers"].copy().merge(
    geo,
    how="left",
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    validate="many_to_one",
    indicator="customer_geo_match",
)
assert len(customer_dim) == len(tables["customers"]), "Customer-geolocation join changed row count"
customer_dim = customer_dim.rename(columns={
    "geolocation_lat": "customer_geo_lat",
    "geolocation_lng": "customer_geo_lng",
    "geolocation_city": "customer_geo_city",
    "geolocation_state": "customer_geo_state",
})
customer_dim["customer_geo_matched"] = customer_dim["customer_geo_match"].eq("both")
customer_dim = customer_dim.drop(columns=["geolocation_zip_code_prefix", "customer_geo_match"])

seller_dim = tables["sellers"].copy().merge(
    geo,
    how="left",
    left_on="seller_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    validate="many_to_one",
    indicator="seller_geo_match",
)
assert len(seller_dim) == len(tables["sellers"]), "Seller-geolocation join changed row count"
seller_dim = seller_dim.rename(columns={
    "geolocation_lat": "seller_geo_lat",
    "geolocation_lng": "seller_geo_lng",
    "geolocation_city": "seller_geo_city",
    "geolocation_state": "seller_geo_state",
})
seller_dim["seller_geo_matched"] = seller_dim["seller_geo_match"].eq("both")
seller_dim = seller_dim.drop(columns=["geolocation_zip_code_prefix", "seller_geo_match"])

customer_geo_match_rate = round(customer_dim["customer_geo_matched"].mean() * 100, 3)
seller_geo_match_rate = round(seller_dim["seller_geo_matched"].mean() * 100, 3)
print(f"Customer geolocation match rate: {customer_geo_match_rate}%")
print(f"Seller geolocation match rate: {seller_geo_match_rate}%")


Customer geolocation match rate: 99.72%
Seller geolocation match rate: 99.774%


## 4. Create enriched order-item fact table

One order can contain several items, products and sellers. We preserve one row per `order_id + order_item_id` and enrich each line with descriptive attributes. Category mapping uses English translation when available, otherwise the original category label; missing categories are explicitly labeled `unknown` and flagged.


In [5]:
order_context_columns = [
    "order_id", "customer_id", "order_status", "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date",
]
order_context = tables["orders"][order_context_columns].merge(
    customer_dim[[c for c in [
        "customer_id", "customer_unique_id", "customer_city", "customer_state", "customer_zip_code_prefix",
        "customer_geo_lat", "customer_geo_lng", "customer_geo_city", "customer_geo_state", "customer_geo_matched"
    ] if c in customer_dim.columns]],
    how="left", on="customer_id", validate="many_to_one"
)
assert len(order_context) == len(tables["orders"]), "Adding customer attributes changed the order row count"

products = tables["products"][[c for c in [
    "product_id", "product_category_name", "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"
] if c in tables["products"].columns]].copy()
products = products.merge(
    tables["category_translation"], how="left", on="product_category_name", validate="many_to_one"
)
products["category_mapping_status"] = np.select(
    [products["product_category_name"].isna(), products["product_category_name_english"].notna()],
    ["missing_source_category", "translated"],
    default="unmapped_source_category",
)
products["product_category_label"] = products["product_category_name_english"].fillna(products["product_category_name"]).fillna("unknown")

item_fact = tables["order_items"].merge(
    order_context[[
        "order_id", "order_status", "order_purchase_timestamp", "order_approved_at",
        "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date",
        "customer_unique_id", "customer_city", "customer_state", "customer_geo_lat", "customer_geo_lng", "customer_geo_matched"
    ]], how="left", on="order_id", validate="many_to_one", indicator="order_match"
)
assert len(item_fact) == len(tables["order_items"]), "Adding order attributes changed item row count"
assert item_fact["order_match"].eq("both").all(), "Some item lines did not match an order"
item_fact = item_fact.drop(columns="order_match")

item_fact = item_fact.merge(products, how="left", on="product_id", validate="many_to_one")
assert len(item_fact) == len(tables["order_items"]), "Adding product attributes changed item row count"

seller_attributes = seller_dim[[c for c in [
    "seller_id", "seller_city", "seller_state", "seller_zip_code_prefix",
    "seller_geo_lat", "seller_geo_lng", "seller_geo_city", "seller_geo_state", "seller_geo_matched"
] if c in seller_dim.columns]]
item_fact = item_fact.merge(seller_attributes, how="left", on="seller_id", validate="many_to_one")
assert len(item_fact) == len(tables["order_items"]), "Adding seller attributes changed item row count"
assert not item_fact.duplicated(["order_id", "order_item_id"]).any(), "Item fact grain was broken by a join"

# Calendar month supports consistent monthly aggregation.
item_fact["order_purchase_month"] = item_fact["order_purchase_timestamp"].dt.to_period("M").astype("string")
item_fact["item_total_including_freight"] = item_fact["price"] + item_fact["freight_value"]

item_fact_columns = [c for c in [
    "order_id", "order_item_id", "order_purchase_month", "order_purchase_timestamp", "order_status",
    "customer_unique_id", "customer_city", "customer_state", "seller_id", "seller_city", "seller_state",
    "product_id", "product_category_name", "product_category_name_english", "product_category_label", "category_mapping_status",
    "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
    "price", "freight_value", "item_total_including_freight", "shipping_limit_date",
    "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date",
    "customer_geo_lat", "customer_geo_lng", "customer_geo_matched", "seller_geo_lat", "seller_geo_lng", "seller_geo_matched"
] if c in item_fact.columns]
item_fact = item_fact[item_fact_columns].copy()
print(f"Order-item fact rows: {len(item_fact):,}; unique item keys: {not item_fact.duplicated(['order_id', 'order_item_id']).any()}")
display(item_fact[[c for c in ["order_id", "order_item_id", "order_status", "customer_state", "seller_state", "product_category_label", "price", "freight_value"] if c in item_fact.columns]].head(5))


Order-item fact rows: 112,650; unique item keys: True


,order_id,order_item_id,order_status,customer_state,seller_state,product_category_label,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,delivered,RJ,SP,cool_stuff,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,delivered,SP,SP,pet_shop,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,delivered,MG,MG,furniture_decor,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,delivered,SP,SP,perfumery,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,delivered,SP,PR,garden_tools,199.90,18.14


## 5. Aggregate items, payments and reviews separately

This is the most important anti-fan-out step. Each aggregation produces no more than one row per `order_id`. We also retain counts so that an order with no matching item/payment/review rows can be distinguished from an order with an actual zero monetary amount.


In [6]:
# Use vectorized groupby sums where possible; this is faster and preserves null when an entire group is null.
item_group = item_fact.groupby("order_id", sort=False)
item_counts = item_group.agg(
    item_count=("order_item_id", "nunique"),
    distinct_product_count=("product_id", "nunique"),
    distinct_seller_count=("seller_id", "nunique"),
    distinct_category_count=("product_category_name", "nunique"),
).reset_index()
item_sums = item_group[["price", "freight_value", "item_total_including_freight"]].sum(min_count=1).reset_index().rename(columns={
    "price": "item_price_total", "freight_value": "freight_value_total"
})
item_unknown = item_fact.assign(_unknown_category=item_fact["category_mapping_status"].isin(["missing_source_category", "unmapped_source_category"]).astype("int64")).groupby("order_id", as_index=False, sort=False)["_unknown_category"].sum().rename(columns={"_unknown_category": "items_with_unknown_category"})
item_agg = item_counts.merge(item_sums, on="order_id", how="left", validate="one_to_one").merge(item_unknown, on="order_id", how="left", validate="one_to_one")

payment_group = tables["payments"].groupby("order_id", sort=False)
payment_agg = payment_group.agg(
    payment_record_count=("payment_sequential", "size"),
    distinct_payment_type_count=("payment_type", "nunique"),
    max_payment_installments=("payment_installments", "max"),
).reset_index()
payment_sums = payment_group[["payment_value"]].sum(min_count=1).reset_index().rename(columns={"payment_value": "payment_value_total"})
payment_agg = payment_agg.merge(payment_sums, on="order_id", how="left", validate="one_to_one")

review_agg = tables["reviews"].groupby("order_id", as_index=False, sort=False).agg(
    review_record_count=("review_id", "size"),
    review_score_count=("review_score", "count"),
    average_review_score=("review_score", "mean"),
    min_review_score=("review_score", "min"),
    max_review_score=("review_score", "max"),
)

for label, frame in [("item_agg", item_agg), ("payment_agg", payment_agg), ("review_agg", review_agg)]:
    assert frame["order_id"].is_unique, f"{label} is not unique by order_id"

print("Rows per aggregated table")
display(pd.DataFrame([
    {"table": "item_agg", "rows": len(item_agg), "unique_order_ids": item_agg.order_id.nunique()},
    {"table": "payment_agg", "rows": len(payment_agg), "unique_order_ids": payment_agg.order_id.nunique()},
    {"table": "review_agg", "rows": len(review_agg), "unique_order_ids": review_agg.order_id.nunique()},
]))


Rows per aggregated table


,table,rows,unique_order_ids
0,item_agg,98666,98666
1,payment_agg,99440,99440
2,review_agg,98673,98673


## 6. Build the one-row-per-order analytical table

The order table remains the left/base table. Aggregate tables are merged using one-to-one validation. Orders with no matching facts stay in the model; their count fields become zero while unobserved monetary and score totals remain null.


In [7]:
order_model = order_context.merge(item_agg, how="left", on="order_id", validate="one_to_one")
order_model = order_model.merge(payment_agg, how="left", on="order_id", validate="one_to_one")
order_model = order_model.merge(review_agg, how="left", on="order_id", validate="one_to_one")

# Record whether a fact aggregation matched before filling only the record-count fields.
order_model["has_item_records"] = order_model["item_count"].notna()
order_model["has_payment_records"] = order_model["payment_record_count"].notna()
order_model["has_review_records"] = order_model["review_record_count"].notna()
for count_col in ["item_count", "distinct_product_count", "distinct_seller_count", "distinct_category_count", "items_with_unknown_category",
                  "payment_record_count", "distinct_payment_type_count", "review_record_count", "review_score_count"]:
    if count_col in order_model.columns:
        order_model[count_col] = order_model[count_col].fillna(0).astype("int64")

# Useful business time fields. Month is calendar month; stage intervals are elapsed hours.
order_model["order_purchase_month"] = order_model["order_purchase_timestamp"].dt.to_period("M").astype("string")

def valid_interval(start: pd.Series, end: pd.Series) -> pd.Series:
    return start.notna() & end.notna() & end.ge(start)

purchase_approval_valid = valid_interval(order_model["order_purchase_timestamp"], order_model["order_approved_at"])
approval_carrier_valid = valid_interval(order_model["order_approved_at"], order_model["order_delivered_carrier_date"])
carrier_delivery_valid = valid_interval(order_model["order_delivered_carrier_date"], order_model["order_delivered_customer_date"])
purchase_delivery_valid = valid_interval(order_model["order_purchase_timestamp"], order_model["order_delivered_customer_date"])

order_model["purchase_to_approval_hours"] = (order_model["order_approved_at"] - order_model["order_purchase_timestamp"]).dt.total_seconds() / 3600
order_model.loc[~purchase_approval_valid, "purchase_to_approval_hours"] = np.nan
order_model["approval_to_carrier_hours"] = (order_model["order_delivered_carrier_date"] - order_model["order_approved_at"]).dt.total_seconds() / 3600
order_model.loc[~approval_carrier_valid, "approval_to_carrier_hours"] = np.nan
order_model["carrier_to_delivery_hours"] = (order_model["order_delivered_customer_date"] - order_model["order_delivered_carrier_date"]).dt.total_seconds() / 3600
order_model.loc[~carrier_delivery_valid, "carrier_to_delivery_hours"] = np.nan

order_model["purchase_to_delivery_elapsed_days"] = (
    order_model["order_delivered_customer_date"] - order_model["order_purchase_timestamp"]
).dt.total_seconds() / 86400
order_model.loc[~purchase_delivery_valid, "purchase_to_delivery_elapsed_days"] = np.nan

purchase_date = order_model["order_purchase_timestamp"].dt.normalize()
actual_delivery_date = order_model["order_delivered_customer_date"].dt.normalize()
estimated_delivery_date = order_model["order_estimated_delivery_date"].dt.normalize()
order_model["purchase_to_delivery_calendar_days"] = (actual_delivery_date - purchase_date).dt.days
order_model.loc[~purchase_delivery_valid, "purchase_to_delivery_calendar_days"] = np.nan
order_model["promised_lead_time_calendar_days"] = (estimated_delivery_date - purchase_date).dt.days
promised_interval_valid = valid_interval(purchase_date, estimated_delivery_date)
order_model.loc[~promised_interval_valid, "promised_lead_time_calendar_days"] = np.nan

order_model["delivery_metric_eligible"] = (
    order_model["order_status"].eq("delivered")
    & purchase_date.notna()
    & actual_delivery_date.notna()
    & estimated_delivery_date.notna()
    & actual_delivery_date.ge(purchase_date)
    & estimated_delivery_date.ge(purchase_date)
)
order_model["delivery_delay_days"] = (actual_delivery_date - estimated_delivery_date).dt.days
order_model.loc[~order_model["delivery_metric_eligible"], "delivery_delay_days"] = np.nan
order_model["is_late_delivery"] = pd.Series(pd.NA, index=order_model.index, dtype="boolean")
eligible_idx = order_model["delivery_metric_eligible"]
order_model.loc[eligible_idx, "is_late_delivery"] = order_model.loc[eligible_idx, "delivery_delay_days"].gt(0)
order_model["delivery_outcome"] = "not_eligible"
order_model.loc[eligible_idx & order_model["is_late_delivery"].fillna(False), "delivery_outcome"] = "late"
order_model.loc[eligible_idx & ~order_model["is_late_delivery"].fillna(False), "delivery_outcome"] = "on_time_or_early"
order_model["order_reviewed_flag"] = order_model["average_review_score"].notna()
order_model["is_low_rated_order"] = pd.Series(pd.NA, index=order_model.index, dtype="boolean")
reviewed_idx = order_model["order_reviewed_flag"]
order_model.loc[reviewed_idx, "is_low_rated_order"] = order_model.loc[reviewed_idx, "average_review_score"].le(2)

order_model["payment_minus_items_and_freight"] = order_model["payment_value_total"] - order_model["item_total_including_freight"]

# Explicitly retain only the intended model columns, in a stable order.
order_model_columns = [c for c in [
    "order_id", "customer_id", "customer_unique_id", "order_status", "order_purchase_month",
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date",
    "customer_city", "customer_state", "customer_zip_code_prefix", "customer_geo_city", "customer_geo_state", "customer_geo_lat", "customer_geo_lng", "customer_geo_matched",
    "has_item_records", "item_count", "distinct_product_count", "distinct_seller_count", "distinct_category_count", "items_with_unknown_category",
    "item_price_total", "freight_value_total", "item_total_including_freight",
    "has_payment_records", "payment_record_count", "payment_value_total", "distinct_payment_type_count", "max_payment_installments", "payment_minus_items_and_freight",
    "has_review_records", "review_record_count", "review_score_count", "average_review_score", "min_review_score", "max_review_score",
    "purchase_to_approval_hours", "approval_to_carrier_hours", "carrier_to_delivery_hours", "purchase_to_delivery_elapsed_days", "purchase_to_delivery_calendar_days", "promised_lead_time_calendar_days",
    "delivery_metric_eligible", "delivery_delay_days", "is_late_delivery", "delivery_outcome", "order_reviewed_flag", "is_low_rated_order"
] if c in order_model.columns]
order_model = order_model[order_model_columns].copy()

assert len(order_model) == len(tables["orders"]), "Order model changed the number of orders"
assert order_model["order_id"].is_unique, "Order model is not one row per order"
assert order_model["order_id"].notna().all(), "Order model has null order IDs"
print(f"Order-level model: {len(order_model):,} rows, {len(order_model.columns)} columns")
display(order_model[[c for c in ["order_id", "order_status", "customer_state", "item_count", "item_price_total", "payment_value_total", "average_review_score", "delivery_outcome", "delivery_delay_days"] if c in order_model.columns]].head(5))


Order-level model: 99,441 rows, 51 columns


,order_id,order_status,customer_state,item_count,item_price_total,payment_value_total,average_review_score,delivery_outcome,delivery_delay_days
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,SP,1,29.99,38.71,4.0,on_time_or_early,-8.0
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,BA,1,118.70,141.46,4.0,on_time_or_early,-6.0
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,GO,1,159.90,179.12,5.0,on_time_or_early,-18.0
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,RN,1,45.00,72.20,5.0,on_time_or_early,-13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,SP,1,19.90,28.62,5.0,on_time_or_early,-10.0


## 7. Automated model validation and financial reconciliation

The model is accepted only if its grain, row preservation, joins and aggregated sums pass structural checks. Payment-versus-item totals are reported as a **diagnostic**, not forced to zero; discounts, adjustments and source-specific definitions may explain differences, and this dataset is not a profit ledger.


In [8]:
validation_rows = []
def add_check(name, observed, expected, passed, notes):
    validation_rows.append({
        "check": name,
        "status": "PASS" if bool(passed) else "FAIL",
        "observed": str(observed),
        "expected": str(expected),
        "notes": notes,
    })

orders_n = len(tables["orders"])
add_check("orders_key_non_null_unique", f"nulls={tables['orders']['order_id'].isna().sum()}, unique={tables['orders']['order_id'].nunique()}", f"nulls=0, unique={orders_n}", tables["orders"]["order_id"].notna().all() and tables["orders"]["order_id"].is_unique, "The orders table defines the target grain.")
add_check("order_model_one_row_per_order", f"rows={len(order_model)}, unique_order_ids={order_model['order_id'].nunique()}", f"rows={orders_n}, unique_order_ids={orders_n}", len(order_model) == orders_n and order_model["order_id"].is_unique, "All orders remain in the order-level model.")
add_check("order_item_row_preservation", f"source={len(tables['order_items'])}, model={len(item_fact)}", f"both={len(tables['order_items'])}", len(item_fact) == len(tables["order_items"]), "The item model retains source line grain.")
add_check("order_item_key_unique", int(item_fact.duplicated(["order_id", "order_item_id"]).sum()), 0, not item_fact.duplicated(["order_id", "order_item_id"]).any(), "Validated after product, order, and seller enrichment.")
add_check("customer_geolocation_join_row_preservation", f"{len(customer_dim)} rows", f"{len(tables['customers'])} rows", len(customer_dim) == len(tables["customers"]), "Left many-to-one lookup; unmatched prefixes remain in the data.")
add_check("seller_geolocation_join_row_preservation", f"{len(seller_dim)} rows", f"{len(tables['sellers'])} rows", len(seller_dim) == len(tables["sellers"]), "Left many-to-one lookup; unmatched prefixes remain in the data.")

for column, source_table, source_column, aggregate_column in [
    ("item_price_total", "order_items", "price", "item_price_total"),
    ("freight_value_total", "order_items", "freight_value", "freight_value_total"),
]:
    source_sum = pd.to_numeric(tables[source_table][source_column], errors="coerce").sum(min_count=1)
    model_sum = order_model[aggregate_column].sum(min_count=1)
    diff = float(model_sum - source_sum) if pd.notna(source_sum) and pd.notna(model_sum) else np.nan
    add_check(f"{column}_sum_reconciliation", f"source={source_sum:.6f}; model={model_sum:.6f}; diff={diff:.10f}", "Difference within 1e-6", pd.notna(diff) and abs(diff) < 1e-6, "Aggregated before joins to avoid join fan-out.")

payment_source_sum = tables["payments"]["payment_value"].sum(min_count=1)
payment_model_sum = order_model["payment_value_total"].sum(min_count=1)
payment_diff = float(payment_model_sum - payment_source_sum) if pd.notna(payment_source_sum) and pd.notna(payment_model_sum) else np.nan
add_check("payment_value_sum_reconciliation", f"source={payment_source_sum:.6f}; model={payment_model_sum:.6f}; diff={payment_diff:.10f}", "Difference within 1e-6", pd.notna(payment_diff) and abs(payment_diff) < 1e-6, "Payments aggregated separately before joining to orders.")

source_review_score_count = int(tables["reviews"]["review_score"].notna().sum())
model_review_score_count = int(order_model["review_score_count"].sum())
add_check("review_score_record_count_reconciliation", model_review_score_count, source_review_score_count, model_review_score_count == source_review_score_count, "Order-level mean scores summarize but do not delete source review records.")

model_validation = pd.DataFrame(validation_rows)
display(model_validation)
model_validation.to_csv(OUTPUT_DIR / "model_validation_report.csv", index=False)

failed_checks = model_validation.loc[model_validation["status"].eq("FAIL")]
if not failed_checks.empty:
    raise AssertionError("Analytical model validation failed:\n" + failed_checks.to_string(index=False))
print("All structural analytical-model validation checks passed.")


,check,status,observed,expected,notes
0,orders_key_non_null_unique,PASS,"nulls=0, unique=99441","nulls=0, unique=99441",The orders table defines the target grain.
1,order_model_one_row_per_order,PASS,"rows=99441, unique_order_ids=99441","rows=99441, unique_order_ids=99441",All orders remain in the order-level model.
2,order_item_row_preservation,PASS,"source=112650, model=112650",both=112650,The item model retains source line grain.
3,order_item_key_unique,PASS,0,0,"Validated after product, order, and seller enr..."
4,customer_geolocation_join_row_preservation,PASS,99441 rows,99441 rows,Left many-to-one lookup; unmatched prefixes re...
5,seller_geolocation_join_row_preservation,PASS,3095 rows,3095 rows,Left many-to-one lookup; unmatched prefixes re...
6,item_price_total_sum_reconciliation,PASS,source=13591643.700000; model=13591643.700000;...,Difference within 1e-6,Aggregated before joins to avoid join fan-out.
7,freight_value_total_sum_reconciliation,PASS,source=2251909.540000; model=2251909.540000; d...,Difference within 1e-6,Aggregated before joins to avoid join fan-out.
8,payment_value_sum_reconciliation,PASS,source=16008872.120000; model=16008872.120000;...,Difference within 1e-6,Payments aggregated separately before joining ...
9,review_score_record_count_reconciliation,PASS,99224,99224,Order-level mean scores summarize but do not d...


All structural analytical-model validation checks passed.


## 8. Diagnostic reports and data dictionary

These reports make assumptions explicit and help future analysts interpret the model correctly.


In [9]:
# Payment-vs-item totals are a diagnostic: do not treat differences automatically as errors or profit.
recon = order_model.loc[order_model["payment_value_total"].notna() & order_model["item_total_including_freight"].notna(),
                        ["order_id", "payment_value_total", "item_total_including_freight", "payment_minus_items_and_freight"]].copy()
recon["absolute_difference"] = recon["payment_minus_items_and_freight"].abs()
no_item_orders = order_model.loc[~order_model["has_item_records"]]
items_without_payments = order_model.loc[order_model["has_item_records"] & ~order_model["has_payment_records"]]
recon_summary = pd.DataFrame([{
    "orders_with_both_payment_and_item_totals": len(recon),
    "orders_difference_within_0_01": int(recon["absolute_difference"].le(0.01).sum()),
    "orders_difference_over_0_01": int(recon["absolute_difference"].gt(0.01).sum()),
    "median_absolute_difference": recon["absolute_difference"].median(),
    "p95_absolute_difference": recon["absolute_difference"].quantile(0.95),
    "total_net_difference_payment_minus_item_freight_matched_orders": recon["payment_minus_items_and_freight"].sum(),
    "orders_with_payment_records_but_no_item_rows": int((no_item_orders["has_payment_records"]).sum()),
    "payment_value_on_orders_without_item_rows": no_item_orders["payment_value_total"].sum(min_count=1),
    "orders_with_item_rows_but_no_payment_records": len(items_without_payments),
    "item_plus_freight_value_on_orders_without_payments": items_without_payments["item_total_including_freight"].sum(min_count=1),
    "interpretation": "Diagnostic only; review order status and possible cancellations/adjustments. Values are not profit.",
}])
recon_summary.to_csv(OUTPUT_DIR / "payment_item_reconciliation_summary.csv", index=False)
display(recon_summary)

missing_item_orders = order_model.loc[~order_model["has_item_records"], [
    "order_id", "order_status", "order_purchase_timestamp", "has_payment_records", "payment_record_count", "payment_value_total"
]].copy()
missing_item_orders["review_note"] = "No matching order-item rows; monetary item totals remain null, not zero. Investigate in context of order status and payment records."
missing_item_orders.to_csv(OUTPUT_DIR / "orders_without_item_records.csv", index=False)


# Field definitions for the business-facing model.
field_definitions = {
    # Order identifiers and customer context
    "order_id": "Unique order identifier; one row per order in order_level_analytics.",
    "customer_id": "Order-specific customer record identifier from Olist.",
    "customer_unique_id": "Customer-level identifier used to recognize repeat customers across order-specific customer_id records.",
    "order_status": "Order lifecycle status recorded by Olist; not an inferred delivery outcome.",
    "order_purchase_month": "Calendar month of the purchase timestamp, formatted YYYY-MM.",
    "order_purchase_timestamp": "Timestamp when the order was placed.",
    "order_approved_at": "Timestamp when payment/order approval was recorded.",
    "order_delivered_carrier_date": "Timestamp when the order was handed to the carrier in the source data.",
    "order_delivered_customer_date": "Timestamp when the order was recorded as delivered to the customer.",
    "order_estimated_delivery_date": "Estimated delivery date promised in the source data; compared at calendar-date level for late-delivery KPIs.",
    "customer_city": "Customer city from the customer table after conservative text normalization.",
    "customer_state": "Customer state abbreviation from the customer table.",
    "customer_zip_code_prefix": "Five-character customer postal-code prefix; not a full postal address.",
    "customer_geo_city": "Most frequent geolocation city label for the customer's postal prefix, with deterministic tie-breaking.",
    "customer_geo_state": "Most frequent geolocation state label for the customer's postal prefix, with deterministic tie-breaking.",
    "customer_geo_lat": "Median geolocation latitude for the customer's postal prefix; approximate area location, not exact address coordinates.",
    "customer_geo_lng": "Median geolocation longitude for the customer's postal prefix; approximate area location, not exact address coordinates.",
    "customer_geo_matched": "True when a geolocation prefix lookup row matched the customer prefix.",
    "seller_id": "Seller identifier from the item-level source table.",
    "seller_city": "Seller city from the seller table.",
    "seller_state": "Seller state abbreviation from the seller table.",
    "seller_zip_code_prefix": "Five-character seller postal-code prefix; not a full postal address.",
    "seller_geo_lat": "Median geolocation latitude for the seller's postal prefix; approximate area location.",
    "seller_geo_lng": "Median geolocation longitude for the seller's postal prefix; approximate area location.",
    "seller_geo_city": "Most frequent geolocation city label for the seller's postal prefix.",
    "seller_geo_state": "Most frequent geolocation state label for the seller's postal prefix.",
    "seller_geo_matched": "True when a geolocation prefix lookup row matched the seller prefix.",
    # Order/item aggregates
    "has_item_records": "True when one or more order-item rows matched. If false, item monetary totals remain null rather than being assumed zero.",
    "item_count": "Number of distinct order item lines; zero means no matching item rows.",
    "distinct_product_count": "Number of distinct product IDs among the order's item lines.",
    "distinct_seller_count": "Number of distinct seller IDs represented by the order's item lines.",
    "distinct_category_count": "Number of distinct non-missing source product categories represented by the order's item lines.",
    "items_with_unknown_category": "Item line count whose category is missing or has no translation mapping.",
    "item_price_total": "Sum of item price values for the order; null if no item aggregate matched.",
    "freight_value_total": "Sum of freight values for the order; null if no item aggregate matched.",
    "item_total_including_freight": "Sum of item price plus freight across item lines; not profit.",
    # Payments and reconciliation
    "has_payment_records": "True when one or more payment rows matched the order.",
    "payment_record_count": "Number of payment records associated with the order.",
    "payment_value_total": "Sum of payment values associated with the order; not net revenue or profit.",
    "distinct_payment_type_count": "Number of distinct payment types recorded for the order.",
    "max_payment_installments": "Maximum installment count among payment rows for the order.",
    "payment_minus_items_and_freight": "Payment total minus item price plus freight total; reconciliation diagnostic, not an accounting verdict.",
    # Reviews
    "has_review_records": "True when at least one review record matched the order, whether or not a score is present.",
    "review_record_count": "Number of review records associated with the order.",
    "review_score_count": "Number of non-missing review scores included in the order-level summary.",
    "average_review_score": "Mean available review score for the order, on a 1–5 scale.",
    "min_review_score": "Lowest available review score associated with the order.",
    "max_review_score": "Highest available review score associated with the order.",
    "order_reviewed_flag": "True when at least one numeric review score is available.",
    "is_low_rated_order": "True when an available order-level mean review score is <= 2; null when no score exists.",
    # Delivery and date-based measures
    "purchase_to_approval_hours": "Elapsed hours from purchase to approval; null for missing or negative intervals.",
    "approval_to_carrier_hours": "Elapsed hours from approval to carrier handoff; null for missing or negative intervals.",
    "carrier_to_delivery_hours": "Elapsed hours from carrier handoff to customer delivery; null for missing or negative intervals.",
    "purchase_to_delivery_elapsed_days": "Elapsed 24-hour days from purchase timestamp to delivery timestamp; null for missing or reversed timestamps.",
    "purchase_to_delivery_calendar_days": "Calendar-day difference between purchase date and actual delivery date.",
    "promised_lead_time_calendar_days": "Calendar-day difference between purchase date and estimated delivery date.",
    "delivery_metric_eligible": "True for delivered orders with purchase, actual delivery and estimated delivery dates in valid calendar sequence.",
    "delivery_delay_days": "Calendar-day difference actual delivery date minus estimated delivery date; positive means late, zero means same-day, negative means early.",
    "is_late_delivery": "True when an eligible order's actual delivery date is later than its estimated delivery date; null for ineligible orders.",
    "delivery_outcome": "late, on_time_or_early, or not_eligible based on the documented calendar-date definition.",
    # Item-grain fields
    "order_item_id": "Item-line sequence identifier within an order; combine with order_id to identify the line.",
    "product_id": "Product identifier for the order item.",
    "product_category_name": "Original normalized Portuguese product category value from the product table; may be missing.",
    "product_category_name_english": "English category label from Olist's category translation lookup; may be missing when unmapped.",
    "product_category_label": "English translated category when available, otherwise source category, otherwise 'unknown'.",
    "category_mapping_status": "Whether category was translated, source category was unmapped, or source category was missing.",
    "product_weight_g": "Recorded product weight in grams.",
    "product_length_cm": "Recorded product length in centimeters.",
    "product_height_cm": "Recorded product height in centimeters.",
    "product_width_cm": "Recorded product width in centimeters.",
    "price": "Item line price from the order-item table, excluding the separately recorded freight value.",
    "freight_value": "Freight value recorded for this item line.",
    "shipping_limit_date": "Seller shipping-limit timestamp recorded for the item.",
    "item_total_including_freight": "Item-line price plus item-line freight value; not a profit measure.",
    "customer_geo_matched": "True when the customer postal prefix matched the geolocation lookup.",
    "seller_geo_matched": "True when the seller postal prefix matched the geolocation lookup.",
}

dictionary_rows = []
for model_name, frame in [("order_level_analytics", order_model), ("order_item_analytics", item_fact)]:
    for column in frame.columns:
        dictionary_rows.append({
            "model": model_name,
            "column": column,
            "dtype": str(frame[column].dtype),
            "definition": field_definitions.get(column, "Descriptive or source-derived field; refer to the original Olist data dictionary for source semantics."),
        })
data_dictionary = pd.DataFrame(dictionary_rows)
data_dictionary.to_csv(OUTPUT_DIR / "analytics_data_dictionary.csv", index=False)
print(f"Data dictionary rows: {len(data_dictionary):,}")


,orders_with_both_payment_and_item_totals,orders_difference_within_0_01,orders_difference_over_0_01,median_absolute_difference,p95_absolute_difference,total_net_difference_payment_minus_item_freight_matched_orders,orders_with_payment_records_but_no_item_rows,payment_value_on_orders_without_item_rows,orders_with_item_rows_but_no_payment_records,item_plus_freight_value_on_orders_without_payments,interpretation
0,98665,98287,378,0.0,2.842171e-14,2870.39,775,162591.95,1,143.46,Diagnostic only; review order status and possi...


Data dictionary rows: 85


## 9. Export the model tables

The two main model files can be consumed by Power BI or SQL tools. Exporting data does not change the source files. If you rerun this notebook, its tables and reports are regenerated.


In [10]:
if EXPORT_ANALYTIC_TABLES:
    order_output = PROCESSED_DIR / "order_level_analytics.csv"
    item_output = PROCESSED_DIR / "order_item_analytics.csv"
    print("Exporting order-level table...", flush=True)
    order_model.to_csv(order_output, index=False, date_format="%Y-%m-%d %H:%M:%S", float_format="%.6f", chunksize=10000)
    print(f"Saved {order_output.relative_to(PROJECT_ROOT)} ({len(order_model):,} rows x {len(order_model.columns)} columns)", flush=True)
    print("Exporting order-item table...", flush=True)
    item_fact.to_csv(item_output, index=False, date_format="%Y-%m-%d %H:%M:%S", float_format="%.6f", chunksize=10000)
    print(f"Saved {item_output.relative_to(PROJECT_ROOT)} ({len(item_fact):,} rows x {len(item_fact.columns)} columns)", flush=True)
else:
    print("EXPORT_ANALYTIC_TABLES=False; model tables were built and validated in memory only.")

print(f"Validation and diagnostic reports saved to: {OUTPUT_DIR.relative_to(PROJECT_ROOT)}")


Exporting order-level table...


Saved data/processed/order_level_analytics.csv (99,441 rows x 51 columns)


Exporting order-item table...


Saved data/processed/order_item_analytics.csv (112,650 rows x 34 columns)


Validation and diagnostic reports saved to: outputs/analytical_model


## 10. How to interpret the outputs

- `data/processed/order_level_analytics.csv` — primary source for order-level KPIs.
- `data/processed/order_item_analytics.csv` — source for product-category, seller and state-level line-item analysis.
- `outputs/analytical_model/model_validation_report.csv` — mandatory structural acceptance checks; all must pass.
- `outputs/analytical_model/payment_item_reconciliation_summary.csv` — differences between paid amount and item-plus-freight totals for investigation.
- `outputs/analytical_model/orders_without_item_records.csv` — orders without item rows; not automatically invalid.
- `outputs/analytical_model/analytics_data_dictionary.csv` — fields and definitions for both models.

### Senior analyst sign-off criteria

Before calculating KPIs, confirm that the validation report contains no `FAIL` rows. Review the payment reconciliation as a diagnostic, and remember that `delivery_metric_eligible` determines the denominator for late delivery rate. The late/on-time comparison must use calendar dates, not full timestamp comparisons against a date-valued estimate.

**Next notebook:** calculate baseline KPIs and run segment analysis only after this model is validated.
